# Pulizia della colonna `history` – NFT Telegram (username e numeri +888)

**Obiettivo.** Ogni riga è un NFT (`address`); la sua `history` è la cronologia degli eventi presa da TonAPI.
Un evento però può contenere **azioni che riguardano altri NFT**: per esempio un trasferimento in blocco
(batch) che sposta 20 NFT insieme, oppure un airdrop. TonAPI restituisce l'evento intero a ogni account coinvolto.

Per ogni azione "estranea" (cioè con `nft` ≠ `address` della riga):

1. si controlla se l'NFT a cui si riferisce (il **target**) è presente nei dataset, anche nell'altro dataset;
2. se il target c'è e **ha già** quella transazione → l'azione si **toglie** dalla riga sbagliata;
3. se il target c'è e **non ha** quella transazione → l'azione si **sposta** nella history del target,
   nella posizione corretta in base a `lt`;
4. se il target **non è** nel dataset (capita spesso, perché questo è un campione) l'azione è *orfana*:
   si toglie dalla riga ma **non si butta**. Va in un CSV a parte, `orphan_nfts_history.csv`, che ha la stessa
   struttura dei dataset puliti: una riga per ogni NFT orfano, con la sua `history`.

Inoltre: `simple_preview` viene eliminato, gli eventi rimasti senza azioni vengono eliminati e gli eventi vengono ordinati per `lt`.

> **Unico parametro: `PATHS`.** Lo stesso codice gira su entrambi i file. Lo spostamento va fatto sui due
> dataset **insieme**, perché alcune azioni di un NFT username compaiono nella history di un numero e viceversa.

## 0. Parametri

In [1]:
import ast, json
from pathlib import Path
import pandas as pd

BASE = Path('/Users/cesareanzellotti/Documents/CNA')
PATHS = {
    'usernames': BASE / 'dataset_sample' / 'fragment_usernames_sample_500.csv',
    'numbers':   BASE / 'dataset_sample' / 'fragment_numbers_sample_500.csv',
}
OUT_DIR = BASE / 'dataset_clean'
ASCENDING = True          # True: dal più vecchio al più recente (ordine cronologico) - False: come TonAPI

pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

def short(addr):
    # Abbrevia un indirizzo TON per le stampe: 0:8f76b983…3b7d
    return addr if not isinstance(addr, str) else f'{addr[:10]}…{addr[-4:]}'

## 1. Caricamento
`history` è una stringa con sintassi Python (`'`, `True`/`False`), quindi si legge con `ast.literal_eval`.
Le history mancanti (NaN) diventano una lista vuota.

In [2]:
def load_dataset(path):
    df = pd.read_csv(path, index_col=0)
    df['events'] = df['history'].apply(lambda s: [] if pd.isna(s) else ast.literal_eval(s))
    return df

datasets = {name: load_dataset(p) for name, p in PATHS.items()}

pd.DataFrame({name: {'righe': len(df),
                     'address duplicati': df.address.duplicated().sum(),
                     'history vuote': (df.events.str.len() == 0).sum(),
                     'eventi totali': df.events.str.len().sum()}
              for name, df in datasets.items()})

,usernames,numbers
righe,5000,5000
address duplicati,0,0
history vuote,104,16
eventi totali,19424,28514


In [3]:
# Un NFT deve comparire in un solo dataset, altrimenti la destinazione di uno spostamento sarebbe ambigua
common = set(datasets['usernames'].address) & set(datasets['numbers'].address)
print('address presenti in entrambi i dataset:', len(common))
assert not common

address presenti in entrambi i dataset: 0


## 2. Com'è fatto un evento
Prendiamo il primo evento della prima riga degli username. Di ogni azione mostriamo solo i campi utili
(senza `simple_preview`) e confrontiamo il campo `nft` con l'`address` della riga.

In [4]:
row0 = datasets['usernames'].iloc[0]
ev0 = row0.events[0]
print('riga   :', row0['name'], short(row0.address))
print('evento :', ev0['event_id'][:16], '| lt', ev0['lt'], '| account', short(ev0['account']['address']),
      '| n. azioni', len(ev0['actions']))
pd.DataFrame([{'type': a['type'],
               'nft': short(a[a['type']]['nft']),
               'sender': (a[a['type']].get('sender') or {}).get('name') or short((a[a['type']].get('sender') or {}).get('address')),
               'recipient': a[a['type']]['recipient'].get('name') or short(a[a['type']]['recipient']['address']),
               'nft == riga?': a[a['type']]['nft'] == row0.address}
              for a in ev0['actions']])

riga   : @katrinakaif 0:8f76b983…3b7d
evento : b325f725f256c75a | lt 67100131000001 | account 0:8f76b983…3b7d | n. azioni 8


,type,nft,sender,recipient,nft == riga?
0,NftItemTransfer,0:8f76b983…3b7d,elector.ton,Getgems Deployer,True
1,NftItemTransfer,0:8f76b983…3b7d,Getgems Deployer,0:71315f35…46a7,True
2,NftItemTransfer,0:e940a345…e1fb,elector.ton,Getgems Deployer,False
3,NftItemTransfer,0:e940a345…e1fb,Getgems Deployer,0:052c75bc…44da,False
4,NftItemTransfer,0:26463e56…0de1,elector.ton,Getgems Deployer,False
5,NftItemTransfer,0:26463e56…0de1,Getgems Deployer,0:398a347d…5697,False
6,NftItemTransfer,0:924eb25c…fc59,elector.ton,Getgems Deployer,False
7,NftItemTransfer,0:924eb25c…fc59,Getgems Deployer,0:beab87d4…a612,False


Questo è un caso tipico: un unico evento (*elector.ton → Getgems Deployer → contratti di vendita*) sposta 4 NFT diversi.
Solo le prime 2 azioni riguardano `@katrinakaif`; le altre 6 appartengono ad altri 3 NFT.

## 3. Una riga per ogni azione
"Esplodiamo" le history in una tabella con una riga per ogni azione. Le colonne `_ev` e `_act` tengono un riferimento
ai dizionari originali, che ci serviranno per ricostruire le history.

**Identificativo della transazione:** `tx_key` = l'elenco degli hash in `base_transactions`.
Così due copie della stessa azione, trovate in righe diverse, hanno la stessa chiave.

In [5]:
def actions_table(df, ds):
    rows = []
    for row_addr, events in zip(df['address'], df['events']):
        for e in events:
            for i, a in enumerate(e['actions']):
                p = a.get(a['type']) or {}
                rows.append({'ds': ds, 'row': row_addr, 'event_id': e['event_id'], 'lt': e['lt'],
                             'action_idx': i, 'n_actions': len(e['actions']), 'type': a['type'],
                             'nft': p.get('nft'),
                             'sender': (p.get('sender') or {}).get('address'),
                             'recipient': (p.get('recipient') or {}).get('address'),
                             'tx_key': ','.join(a['base_transactions']),
                             '_ev': e, '_act': a})
    return pd.DataFrame(rows)

A = pd.concat([actions_table(df, ds) for ds, df in datasets.items()], ignore_index=True)
# Un'azione senza campo `nft` (altri tipi di azione) non si può attribuire a un altro NFT: resta dov'è
A['foreign'] = A['nft'].notna() & (A['nft'] != A['row'])
VIEW = [c for c in A.columns if not c.startswith('_')]

print('tipi di azione:', A['type'].value_counts().to_dict())
print('azioni senza campo nft:', A['nft'].isna().sum())
A[VIEW].head(8).assign(**{c: (lambda d, c=c: d[c].map(short)) for c in ['row', 'nft', 'sender', 'recipient']},
                       event_id=lambda d: d.event_id.str[:12], tx_key=lambda d: d.tx_key.str[:12] + '…')

tipi di azione: {'NftItemTransfer': 111728}
azioni senza campo nft: 0


,ds,row,event_id,lt,action_idx,n_actions,type,nft,sender,recipient,tx_key,foreign
0,usernames,0:8f76b983…3b7d,b325f725f256,67100131000001,0,8,NftItemTransfer,0:8f76b983…3b7d,0:e1022547…15e9,0:39d63083…7e0e,c6a32779bb85…,False
1,usernames,0:8f76b983…3b7d,b325f725f256,67100131000001,1,8,NftItemTransfer,0:8f76b983…3b7d,0:39d63083…7e0e,0:71315f35…46a7,2b838c1b77ea…,False
2,usernames,0:8f76b983…3b7d,b325f725f256,67100131000001,2,8,NftItemTransfer,0:e940a345…e1fb,0:e1022547…15e9,0:39d63083…7e0e,a7726fe4e175…,True
3,usernames,0:8f76b983…3b7d,b325f725f256,67100131000001,3,8,NftItemTransfer,0:e940a345…e1fb,0:39d63083…7e0e,0:052c75bc…44da,7d72c08a6606…,True
4,usernames,0:8f76b983…3b7d,b325f725f256,67100131000001,4,8,NftItemTransfer,0:26463e56…0de1,0:e1022547…15e9,0:39d63083…7e0e,a23345e778c9…,True
5,usernames,0:8f76b983…3b7d,b325f725f256,67100131000001,5,8,NftItemTransfer,0:26463e56…0de1,0:39d63083…7e0e,0:398a347d…5697,6975652ceab0…,True
6,usernames,0:8f76b983…3b7d,b325f725f256,67100131000001,6,8,NftItemTransfer,0:924eb25c…fc59,0:e1022547…15e9,0:39d63083…7e0e,648c2b0834f9…,True
7,usernames,0:8f76b983…3b7d,b325f725f256,67100131000001,7,8,NftItemTransfer,0:924eb25c…fc59,0:39d63083…7e0e,0:beab87d4…a612,7ff1eb041f32…,True


In [6]:
# Controllo della chiave: ogni tx_key deve corrispondere a un solo evento e a un solo NFT
k = A.groupby('tx_key').agg(n_event=('event_id', 'nunique'), n_nft=('nft', 'nunique'))
print('tx_key con più di un evento:', (k.n_event > 1).sum(), '| con più di un NFT:', (k.n_nft > 1).sum())
assert (k.n_event == 1).all() and (k.n_nft == 1).all()

# Le azioni "proprie" (nft == riga) non devono mai essere duplicate in un'altra riga
own_dup = A[~A.foreign].tx_key.duplicated().sum()
print('azioni proprie duplicate tra righe:', own_dup)

tx_key con più di un evento: 0 | con più di un NFT: 0
azioni proprie duplicate tra righe: 0


## 4. Quante azioni estranee ci sono?

In [7]:
summary = A.groupby('ds').agg(azioni=('foreign', 'size'), estranee=('foreign', 'sum'))
summary['% estranee'] = (100 * summary.estranee / summary.azioni).round(1)
summary

,azioni,estranee,% estranee
ds,,,
numbers,77133,45810,59.4
usernames,34595,13573,39.2


In [8]:
# Classificazione degli eventi in base a quante azioni estranee contengono
ev = A.groupby(['ds', 'row', 'event_id']).agg(n=('foreign', 'size'), nf=('foreign', 'sum')).reset_index()
ev['tipo'] = 'solo proprie'
ev.loc[(ev.nf > 0) & (ev.nf < ev.n), 'tipo'] = 'miste'
ev.loc[ev.nf == ev.n, 'tipo'] = 'solo estranee'
pd.crosstab(ev.ds, ev.tipo, margins=True)

tipo,miste,solo estranee,solo proprie,All
ds,,,,
numbers,1937,363,26214,28514
usernames,889,76,18459,19424
All,2826,439,44673,47938


Gli eventi **"solo estranee"** non contengono nessuna azione sul NFT della riga. Di solito compaiono nella history perché
l'indirizzo del NFT è *sender* o *recipient* di un altro NFT: tipicamente un airdrop mandato a migliaia di indirizzi.
Dopo la pulizia diventano vuoti e vengono eliminati.

In [9]:
F = A[A.foreign].copy()
F['row_is_party'] = (F.sender == F.row) | (F.recipient == F.row)
only_foreign = ev.loc[ev.tipo == 'solo estranee', ['row', 'event_id']]
x = F.merge(only_foreign, on=['row', 'event_id'])
print('eventi "solo estranee" in cui la riga è sender/recipient di un altro NFT:',
      x.groupby(['row', 'event_id']).row_is_party.any().sum(), 'su', len(only_foreign))

eventi "solo estranee" in cui la riga è sender/recipient di un altro NFT: 409 su 439


## 5. La transazione è già presente altrove nel dataset?
Per ogni azione estranea calcoliamo:
* `target_ds`: il dataset in cui si trova l'NFT target (`None` se non c'è);
* `in_target`: se la history del target contiene **già** la stessa transazione (stesso `tx_key`) come azione propria;
* `copies`: in quante righe compare la stessa azione estranea. Un evento batch infatti si ripete nella history di ogni NFT coinvolto.

In [10]:
addr2ds = {a: ds for ds, df in datasets.items() for a in df.address}
own_keys = set(zip(A.loc[~A.foreign, 'row'], A.loc[~A.foreign, 'tx_key']))

F['target_ds'] = F.nft.map(addr2ds)
F['in_target'] = [(n, k) in own_keys for n, k in zip(F.nft, F.tx_key)]
F['copies'] = F.groupby('tx_key').row.transform('nunique')

def classify(r):
    if r.target_ds is None or pd.isna(r.target_ds):
        return 'orfana (target fuori dal dataset) -> salva a parte'
    return 'gia nel target -> rimuovi' if r.in_target else 'manca nel target -> sposta'

F['esito'] = F.apply(classify, axis=1)
pd.crosstab([F.ds, F.target_ds.fillna('-')], F.esito, margins=True)

esito                gia nel target -> rimuovi  manca nel target -> sposta  orfana (target fuori dal dataset) -> salva a parte    All
ds        target_ds                                                                                                                  
numbers   -                                  0                           0                                               44562  44562
          numbers                         1220                          21                                                   0   1241
          usernames                          7                           0                                                   0      7
usernames -                                  0                           0                                               13439  13439
          numbers                            7                           0                                                   0      7
          usernames                        126                           1                                                   0    127
All                                       1360                          22                                               58001  59383

In [11]:
print('azioni estranee (righe del log):', len(F), '| transazioni estranee distinte:', F.tx_key.nunique())
print('\nIn quante righe compare la stessa azione estranea:')
F.drop_duplicates('tx_key').copies.value_counts().sort_index().head(10)

azioni estranee (righe del log): 59383 | transazioni estranee distinte: 29079

In quante righe compare la stessa azione estranea:


copies
1     19356
2      3946
3      1309
4      2206
5       790
6       724
7       109
8       184
16      105
17       70
Name: count, dtype: int64

### 5a. Esempio di azione da spostare (il caso "0x35 contiene una transazione di 0x40")

In [12]:
move = F[F.esito.str.startswith('manca')]
# come esempio prendiamo un target che ha già qualche evento, così si vede in che posizione finisce quello spostato
n_ev = A.groupby('row').event_id.nunique()
cand = move.assign(n_ev_target=move.nft.map(n_ev).fillna(0))
ex = cand[cand.n_ev_target.between(2, 8)].iloc[0] if cand.n_ev_target.between(2, 8).any() else cand.iloc[0]
name_of = {a: n for df in datasets.values() for a, n in zip(df.address, df.name)}
print(f"riga sorgente : {name_of[ex.row]:20s} {short(ex.row)}  ({ex.ds})")
print(f"NFT dell'azione: {name_of[ex.nft]:20s} {short(ex.nft)}  ({ex.target_ds})")
print(f"evento {ex.event_id[:16]}…  lt={ex['lt']}  azioni nell'evento={ex.n_actions}")
print('\nEventi del target PRIMA (lt):', [e['lt'] for e in datasets[ex.target_ds].set_index('address').loc[ex.nft, 'events']])

riga sorgente : +888 0640 5273       0:03a719d5…ec5a  (numbers)
NFT dell'azione: +888 0642 7574       0:756ccaff…536a  (numbers)
evento 2da790a03e0dfe6f…  lt=64254695000001  azioni nell'evento=100

Eventi del target PRIMA (lt): [65859037000001, 65853716000001]


In [13]:
# Tutte le azioni da spostare (con le copie in più righe)
move[['ds', 'row', 'nft', 'target_ds', 'event_id', 'lt', 'copies']].assign(
    row=lambda d: d.row.map(name_of), nft=lambda d: d.nft.map(name_of), event_id=lambda d: d.event_id.str[:12])

,ds,row,nft,target_ds,event_id,lt,copies
15667,usernames,@qk777,@onlinenow,usernames,02602095ee6e,54178279000001,1
36214,numbers,+888 0340 6915,+888 0694 0793,numbers,39bfaa8ce8af,42514415000001,1
36215,numbers,+888 0340 6915,+888 0694 0793,numbers,39bfaa8ce8af,42514415000001,1
39759,numbers,+888 0640 5273,+888 0642 7574,numbers,2da790a03e0d,64254695000001,5
46309,numbers,+888 0725 9218,+888 0642 7574,numbers,2da790a03e0d,64254695000001,5
54017,numbers,+888 0369 4721,+888 0369 1926,numbers,4bfb6388e6c0,42202710000001,1
60227,numbers,+888 0597 6142,+888 0642 7574,numbers,2da790a03e0d,64254695000001,5
65515,numbers,+888 0313 2465,+888 0704 6253,numbers,fb111f5b3509,60792972000001,1
65516,numbers,+888 0313 2465,+888 0704 6253,numbers,fb111f5b3509,60792972000001,1
66682,numbers,+888 0624 0373,+888 0552 4293,numbers,1bcb0faa32a0,44258292000001,1


### 5b. Azioni tra dataset diversi (username ↔ numeri)

In [14]:
cross = F[F.target_ds.notna() & (F.target_ds != F.ds)]
cross[['ds', 'row', 'target_ds', 'nft', 'esito', 'lt']].assign(row=lambda d: d.row.map(name_of), nft=lambda d: d.nft.map(name_of))

,ds,row,target_ds,nft,esito,lt
862,usernames,@usd005,numbers,+888 0479 2651,gia nel target -> rimuovi,49498774000001
6419,usernames,@da888,numbers,+888 0621 7390,gia nel target -> rimuovi,67656616000001
6430,usernames,@da888,numbers,+888 0621 7390,gia nel target -> rimuovi,66739387000001
15282,usernames,@xiaoe,numbers,+888 8 688,gia nel target -> rimuovi,68675196000001
16892,usernames,@gongfang,numbers,+888 8 688,gia nel target -> rimuovi,68675196000001
24492,usernames,@xxll,numbers,+888 0596 7213,gia nel target -> rimuovi,66285712000001
29069,usernames,@xinbisaiche,numbers,+888 8 688,gia nel target -> rimuovi,68675196000001
70571,numbers,+888 0596 7213,usernames,@xxll,gia nel target -> rimuovi,66285712000001
76464,numbers,+888 0621 7390,usernames,@da888,gia nel target -> rimuovi,67656616000001
76475,numbers,+888 0621 7390,usernames,@da888,gia nel target -> rimuovi,66739387000001


## 6. Ricostruzione delle history
Ogni NFT riceve:
* le sue azioni proprie (`nft == address`);
* le azioni spostate da altre righe, **una sola volta** anche se compaiono in più righe (deduplica su `tx_key`).

Poi le azioni vengono raggruppate per `event_id`. Ogni evento conserva i suoi metadati (`event_id`, `timestamp`, `lt`, `is_scam`, …);
per gli eventi spostati `account.address` diventa quello del target. `simple_preview` viene eliminato.
Infine gli eventi vengono ordinati per `lt`.
L'`lt` di un evento è lo stesso in tutte le righe in cui compare: lo verifichiamo qui sotto, perché garantisce che l'evento spostato vada nella posizione giusta.

In [15]:
shared = A.groupby('event_id').agg(n_row=('row', 'nunique'), n_lt=('lt', 'nunique'))
print('eventi presenti in più righe:', (shared.n_row > 1).sum(), '| con lt diverso tra le righe:', (shared.n_lt > 1).sum())
assert (shared.n_lt == 1).all()

eventi presenti in più righe: 230 | con lt diverso tra le righe: 0


In [16]:
keep  = A[~A.foreign].assign(dest=lambda d: d.row)
moved = move.drop_duplicates('tx_key').assign(dest=lambda d: d.nft)
final = pd.concat([keep, moved]).drop_duplicates(['dest', 'tx_key'])
print(f'azioni proprie tenute: {len(keep)} | spostate (distinte): {len(moved)} | totale finale: {len(final)}')

def clean_action(a):
    return {k: v for k, v in a.items() if k != 'simple_preview'}

def build_history(addr, acts):
    events = []
    for _, g in acts.sort_values('action_idx').groupby('event_id', sort=False):
        e = g['_ev'].iloc[0]
        new = {k: v for k, v in e.items() if k != 'actions'}
        new['account'] = {**e['account'], 'address': addr}
        new['actions'] = [clean_action(a) for a in g['_act']]
        events.append(new)
    return sorted(events, key=lambda e: e['lt'], reverse=not ASCENDING)

by_dest = dict(tuple(final.groupby('dest')))
clean = {}
for ds, df in datasets.items():
    out = df[['address', 'name', 'owner_address']].copy()
    out['events'] = [build_history(a, by_dest[a]) if a in by_dest else [] for a in df.address]
    clean[ds] = out

azioni proprie tenute: 52345 | spostate (distinte): 16 | totale finale: 52361


### 6b. Le azioni orfane
Le azioni orfane non vengono buttate: costruiamo con la stessa `build_history` una tabella con **una riga per ogni NFT orfano**.
Ogni azione compare una sola volta, anche se era in più righe. In più:
* `found_in` è l'elenco delle righe in cui l'azione è stata trovata;
* `datasets` dice da quale dataset arrivano.

Con il dataset completo molti di questi NFT avranno una loro riga e le azioni verranno spostate lì invece che finire qui.

In [17]:
orph = F[F.esito.str.startswith('orfana')]
orph_u = orph.drop_duplicates('tx_key').assign(dest=lambda d: d.nft)

orphans = (orph.groupby('nft')
               .agg(found_in=('row', lambda s: sorted(set(s))), datasets=('ds', lambda s: ','.join(sorted(set(s)))))
               .rename_axis('address').reset_index())
orph_by_nft = dict(tuple(orph_u.groupby('nft')))
orphans['events'] = [build_history(a, orph_by_nft[a]) for a in orphans.address]

print(f'azioni orfane: {len(orph)} (righe) -> {len(orph_u)} distinte | NFT orfani: {len(orphans)}')
orphans.assign(address=lambda d: d.address.map(short),
               n_righe_sorgente=lambda d: d.found_in.str.len(),
               n_eventi=lambda d: d.events.str.len(),
               n_azioni=lambda d: d.events.apply(lambda h: sum(len(e['actions']) for e in h))
              ).drop(columns=['found_in', 'events']).sort_values('n_azioni', ascending=False).head(10)

azioni orfane: 58001 (righe) -> 28426 distinte | NFT orfani: 18112


,address,datasets,n_righe_sorgente,n_eventi,n_azioni
9782,0:899f3a2e…ad5a,numbers,1,59,88
11779,0:a68153f7…3540,numbers,1,22,33
8607,0:79842886…5ead,numbers,2,19,29
2738,0:27480639…f69e,usernames,1,18,26
10460,0:93e4048c…fcdc,usernames,1,14,23
414,0:062371ed…5d75,usernames,1,14,21
9309,0:8356fe2c…8d81,numbers,5,13,20
9349,0:83eed994…a02f,numbers,3,12,18
12787,0:b40ed4dd…3633,numbers,3,12,18
5422,0:4d4a5b01…0afd,numbers,5,12,18


## 7. Verifiche

In [18]:
C = pd.concat([actions_table(df, ds) for ds, df in clean.items()], ignore_index=True)
O = actions_table(orphans, 'orphans')

checks = {
    'ogni azione riguarda il NFT della riga':          ((C.nft == C.row) | C.nft.isna()).all(),
    'nessuna transazione in più di una riga':          not C.tx_key.duplicated().any(),
    'tutte le azioni proprie originali sono rimaste':  set(keep.tx_key) <= set(C.tx_key),
    'tutte le azioni da spostare sono nel target':     set(zip(moved.nft, moved.tx_key)) <= set(zip(C.row, C.tx_key)),
    'n. azioni = proprie + spostate':                  len(C) == len(keep) + len(moved),
    'nessun evento vuoto':                             all(len(e['actions']) for df in clean.values() for h in df.events for e in h),
    'nessun simple_preview':                           not any('simple_preview' in a for a in C._act),
    'account.address == riga':                         all(e['account']['address'] == a for df in clean.values() for a, h in zip(df.address, df.events) for e in h),
    'eventi ordinati per lt':                          all([e['lt'] for e in h] == sorted([e['lt'] for e in h], reverse=not ASCENDING) for df in clean.values() for h in df.events),
    # orfane
    'orfane: ogni azione riguarda il NFT della riga':  ((O.nft == O.row)).all(),
    'orfane: ogni transazione una sola volta':          not O.tx_key.duplicated().any(),
    'orfane: nessun NFT orfano è nei dataset':          not set(O.row) & set(addr2ds),
    'nessuna transazione persa (puliti + orfane = originali)': set(C.tx_key) | set(O.tx_key) == set(A.tx_key),
    'nessuna transazione sia nei puliti che nelle orfane':      not set(C.tx_key) & set(O.tx_key),
}
for k, v in checks.items():
    print('OK ' if v else 'KO ', k)
assert all(checks.values())

OK  ogni azione riguarda il NFT della riga
OK  nessuna transazione in più di una riga
OK  tutte le azioni proprie originali sono rimaste
OK  tutte le azioni da spostare sono nel target
OK  n. azioni = proprie + spostate
OK  nessun evento vuoto
OK  nessun simple_preview
OK  account.address == riga
OK  eventi ordinati per lt
OK  orfane: ogni azione riguarda il NFT della riga
OK  orfane: ogni transazione una sola volta
OK  orfane: nessun NFT orfano è nei dataset
OK  nessuna transazione persa (puliti + orfane = originali)
OK  nessuna transazione sia nei puliti che nelle orfane


In [19]:
pd.DataFrame({'eventi prima': A.drop_duplicates(['row', 'event_id']).groupby('ds').size(),
              'eventi dopo':  C.drop_duplicates(['row', 'event_id']).groupby('ds').size(),
              'azioni prima': A.groupby('ds').size(),
              'azioni dopo':  C.groupby('ds').size(),
              'history vuote prima': pd.Series({ds: (df.events.str.len() == 0).sum() for ds, df in datasets.items()}),
              'history vuote dopo':  pd.Series({ds: (df.events.str.len() == 0).sum() for ds, df in clean.items()}),
              'azioni orfane salvate a parte': orph_u.groupby('ds').size()})

,eventi prima,eventi dopo,azioni prima,azioni dopo,history vuote prima,history vuote dopo,azioni orfane salvate a parte
numbers,28514,28162,77133,31338,16,16,16820
usernames,19424,19349,34595,21023,104,103,11606


### Prima / dopo per l'esempio del punto 5a

In [20]:
def show_history(addr, events):
    rows = []
    for e in events:
        nfts = {a[a['type']]['nft'] for a in e['actions']}
        rows.append({'lt': e['lt'], 'event_id': e['event_id'][:12], 'n azioni': len(e['actions']),
                     'NFT diversi': len(nfts), 'solo questo NFT': nfts == {addr},
                     'evento dell\'esempio': '<<<' if e['event_id'] == ex.event_id else ''})
    return pd.DataFrame(rows)

for label, addr in [('SORGENTE', ex.row), ('TARGET', ex.nft)]:
    ds = addr2ds[addr]
    print(f'--- {label}: {name_of[addr]} ({ds}) — PRIMA')
    display(show_history(addr, datasets[ds].set_index('address').loc[addr, 'events']))
    print(f'--- {label}: {name_of[addr]} ({ds}) — DOPO')
    display(show_history(addr, clean[ds].set_index('address').loc[addr, 'events']))

--- SORGENTE: +888 0640 5273 (numbers) — PRIMA


,lt,event_id,n azioni,NFT diversi,solo questo NFT,evento dell'esempio
0,64644271000001,07904dc02e8f,1,1,True,
1,64575479000001,70432b45183c,4,4,False,
2,64254695000001,2da790a03e0d,100,100,False,<<<
3,63738141000001,b781bf8415b0,1,1,True,
4,33686156000001,0aad19e7664f,1,1,True,
5,33620883000001,50ac49c7e6c4,1,1,True,


--- SORGENTE: +888 0640 5273 (numbers) — DOPO


,lt,event_id,n azioni,NFT diversi,solo questo NFT,evento dell'esempio
0,33620883000001,50ac49c7e6c4,1,1,True,
1,33686156000001,0aad19e7664f,1,1,True,
2,63738141000001,b781bf8415b0,1,1,True,
3,64254695000001,2da790a03e0d,1,1,True,<<<
4,64575479000001,70432b45183c,1,1,True,
5,64644271000001,07904dc02e8f,1,1,True,


--- TARGET: +888 0642 7574 (numbers) — PRIMA


,lt,event_id,n azioni,NFT diversi,solo questo NFT,evento dell'esempio
0,65859037000001,82616b8be401,1,1,True,
1,65853716000001,b5a1a8cbcfcd,1,1,True,


--- TARGET: +888 0642 7574 (numbers) — DOPO


,lt,event_id,n azioni,NFT diversi,solo questo NFT,evento dell'esempio
0,64254695000001,2da790a03e0d,1,1,True,<<<
1,65853716000001,b5a1a8cbcfcd,1,1,True,
2,65859037000001,82616b8be401,1,1,True,


## 8. Metriche prima / dopo la pulizia
Le metriche sono calcolate sulle `history`:
* **prima** = dataset originali. Un evento batch viene contato in **ogni** riga in cui compare, insieme alle azioni degli altri NFT;
* **dopo** = dataset puliti, dove ogni azione sta solo nella riga del suo NFT;
* **orfane** = il file a parte con gli NFT che non sono nei dataset.

Gli indirizzi sono i `sender` e `recipient` delle azioni, contati una volta sola. Con `is_wallet=False` si intende un contratto
(marketplace, contratto di vendita, collezione), non una persona. Un *arco* è una coppia distinta sender → recipient.

In [21]:
def metrics(df):
    events = [e for h in df.events for e in h]
    acts = [(a['status'], a[a['type']]) for e in events for a in e['actions']]
    parties, edges, scam_acts = {}, set(), 0
    for _, p in acts:
        snd, rcp = p.get('sender'), p.get('recipient')
        for q in (snd, rcp):
            if q:
                parties[q['address']] = q
        scam_acts += bool((snd and snd['is_scam']) or (rcp and rcp['is_scam']))
        if snd and rcp:
            edges.add((snd['address'], rcp['address'], snd['is_wallet'] and rcp['is_wallet']))
    P = parties.values()
    return pd.Series({
        'NFT (righe)':                            len(df),
        'history vuote':                          int((df.events.str.len() == 0).sum()),
        'eventi':                                 len(events),
        'eventi per NFT (media)':                 round(len(events) / max(len(df), 1), 2),
        'azioni':                                 len(acts),
        'azioni per evento (media)':              round(len(acts) / max(len(events), 1), 2),
        'eventi con più NFT (batch)':             sum(len({a[a['type']]['nft'] for a in e['actions']}) > 1 for e in events),
        'eventi is_scam':                         sum(e['is_scam'] for e in events),
        'azioni con sender/recipient scam':       scam_acts,
        'azioni failed':                          sum(st != 'ok' for st, _ in acts),
        'azioni senza sender (mint)':             sum(not p.get('sender') for _, p in acts),
        'azioni con commento':                    sum(bool(p.get('comment')) for _, p in acts),
        'azioni con commento cifrato':            sum('encrypted_comment' in p for _, p in acts),
        'indirizzi distinti':                     len(P),
        '  di cui wallet (is_wallet=True)':       sum(q['is_wallet'] for q in P),
        '  di cui contratti (is_wallet=False)':   sum(not q['is_wallet'] for q in P),
        '  di cui scam (is_scam=True)':           sum(q['is_scam'] for q in P),
        '  di cui con nome (.ton, .t.me, …)':     sum('name' in q for q in P),
        'archi distinti sender → recipient':      len({(a, b) for a, b, _ in edges}),
        '  di cui wallet → wallet':               sum(w for _, _, w in edges),
    }, dtype=object)

cols = {}
for ds in datasets:
    cols[(ds, 'prima')] = metrics(datasets[ds])
    cols[(ds, 'dopo')]  = metrics(clean[ds])
cols[('totale', 'prima')] = metrics(pd.concat(datasets.values()))
cols[('totale', 'dopo')]  = metrics(pd.concat(clean.values()))
before, after = cols[('totale', 'prima')], cols[('totale', 'dopo')]
cols[('totale', 'differenza')] = pd.Series({k: round(after[k] - before[k], 2) for k in before.index}, dtype=object)
cols[('orfane', '')] = metrics(orphans)
M = pd.DataFrame(cols)
M

usernames        numbers         totale                   orfane
                                         prima   dopo   prima   dopo   prima   dopo differenza       
NFT (righe)                               5000   5000    5000   5000   10000  10000          0  18112
history vuote                              104    103      16     16     120    119         -1      0
eventi                                   19424  19349   28514  28162   47938  47511       -427  24129
eventi per NFT (media)                    3.88   3.87     5.7   5.63    4.79   4.75      -0.04   1.33
azioni                                   34595  21023   77133  31338  111728  52361     -59367  28426
azioni per evento (media)                 1.78   1.09    2.71   1.11    2.33    1.1      -1.23   1.18
eventi con più NFT (batch)                 939      0    2270      0    3209      0      -3209      0
eventi is_scam                             367    304     347     31     714    335       -379   2410
azioni con sender/recipient scam             0      0       0      0       0      0          0      0
azioni failed                              345    324    1147    581    1492    905       -587    233
azioni senza sender (mint)                5055   5037    1010   1002    6065   6039        -26     26
azioni con commento                      12964   8604   24499  12993   37463  21597     -15866   6967
azioni con commento cifrato                  9      9       5      5      14     14          0      0
indirizzi distinti                       18119  12861   18130  14303   35527  26462      -9065  10249
  di cui wallet (is_wallet=True)         11282  10216   11576  10657   22152  20187      -1965   3045
  di cui contratti (is_wallet=False)      6837   2645    6554   3646   13375   6275      -7100   7204
  di cui scam (is_scam=True)                 0      0       0      0       0      0          0      0
  di cui con nome (.ton, .t.me, …)        1866   1845    1166   1145    2815   2777        -38    306
archi distinti sender → recipient        19532  13954   25303  20620   44601  34340     -10261  11201
  di cui wallet → wallet                  9275   8272   10910   9789   19995  17871      -2124   2638

**Come leggere la tabella (valori del campione):**
* **eventi con più NFT (batch)**: da 3.209 a 0. Dopo la pulizia ogni evento riguarda un solo NFT.
* **azioni**: da 111.728 a 52.361 nei dataset, più 28.426 nel file delle orfane. Le restanti 30.941 erano **copie** della stessa transazione, ripetute nelle history di più NFT.
* **eventi is_scam**: da 714 a 335. Nei numeri si passa da 347 a 31: quasi tutti gli eventi scam erano airdrop di *altri* NFT (spam) mandati all'indirizzo del numero, e ora stanno nelle orfane. Lì sono 2.410 perché un airdrop viene diviso in un evento per ogni NFT.
* **indirizzi**: calano molto di più i contratti (13.375 → 6.275) che i wallet (22.152 → 20.187). Le azioni estranee passavano soprattutto dai contratti di vendita degli altri NFT.
* **is_scam sugli indirizzi** (sender/recipient) è sempre `False` in questo campione: TonAPI segna lo scam solo a livello di evento.

## 9. Salvataggio
* `<nome>_clean.csv`: stesse colonne dell'originale; `history` ora è **JSON** (si legge con `json.loads`, non serve più `ast`);
* `orphan_nfts_history.csv`: gli NFT orfani con la loro `history` (JSON), più `found_in` (JSON) e `datasets`;
* `foreign_actions_log.csv`: tutte le azioni estranee tolte dalle righe, con l'esito (spostata / già presente / orfana) e il numero di copie.

In [22]:
OUT_DIR.mkdir(exist_ok=True)
for ds, df in clean.items():
    out = df.drop(columns='events').assign(history=df.events.apply(json.dumps))
    path = OUT_DIR / f'{Path(PATHS[ds]).stem}_clean.csv'
    out.to_csv(path)
    print('salvato', path)

path = OUT_DIR / 'orphan_nfts_history.csv'
orphans.drop(columns='events').assign(found_in=orphans.found_in.apply(json.dumps),
                                      history=orphans.events.apply(json.dumps)).to_csv(path, index=False)
print('salvato', path, f'({len(orphans)} NFT orfani)')

log = F[['ds', 'row', 'event_id', 'lt', 'action_idx', 'nft', 'target_ds', 'in_target', 'copies', 'row_is_party', 'esito', 'tx_key']]
log.to_csv(OUT_DIR / 'foreign_actions_log.csv', index=False)
print('salvato', OUT_DIR / 'foreign_actions_log.csv', f'({len(log)} righe)')

salvato /Users/cesareanzellotti/Documents/CNA/dataset_clean/fragment_usernames_sample_500_clean.csv


salvato /Users/cesareanzellotti/Documents/CNA/dataset_clean/fragment_numbers_sample_500_clean.csv


salvato /Users/cesareanzellotti/Documents/CNA/dataset_clean/orphan_nfts_history.csv (18112 NFT orfani)


salvato /Users/cesareanzellotti/Documents/CNA/dataset_clean/foreign_actions_log.csv (59383 righe)


In [23]:
# Rilettura di controllo
chk = pd.read_csv(OUT_DIR / f"{Path(PATHS['usernames']).stem}_clean.csv", index_col=0)
h = json.loads(chk.history.iloc[0])
print(chk.shape, '| eventi riga 0:', len(h), '| chiavi evento:', list(h[0]))

orf = pd.read_csv(OUT_DIR / 'orphan_nfts_history.csv')
print(orf.shape, list(orf.columns))
o0 = orf.iloc[0]
print('esempio:', o0.address, '| trovato in', json.loads(o0.found_in), '| eventi:', len(json.loads(o0.history)))

(5000, 4) | eventi riga 0: 3 | chiavi evento: ['event_id', 'account', 'timestamp', 'is_scam', 'lt', 'in_progress', 'extra', 'progress', 'actions']


(18112, 4) ['address', 'found_in', 'datasets', 'history']
esempio: 0:0005bc552678bcff576db977c2c06b926842bde3ff6a3859d8da49fc131cf94c | trovato in ['0:fde7fde441704b5f8a535681f462a03ec3de61655b00023a7b027f3111562334'] | eventi: 1


## 10. Categorie Getgems per i numeri

Getgems assegna ai numeri 34 categorie, tutte basate su regole delle cifre. Le regole qui sotto sono state ricavate dagli esempi
delle pagine delle categorie e poi verificate sul sito (sezione 10b). `s` sono le cifre dopo il prefisso `888`: 8 nei numeri
normali, 4 nei numeri corti `+888 8 xxx`.

* **Ripetizioni**: almeno N cifre uguali di fila. Le categorie sono cumulative: `0000` sta in 2, 3 e 4 Digits.
* **Schemi** (ABAB, AABB, ABCABC…): in qualsiasi punto, con lettere = cifre diverse. In AxAxA la `x` è una cifra qualsiasi.
* **Club** (69, 666, 777, 888, 520, 168, 1314, 123, 007) e **Pi Club** (`31415`): `s` contiene la sequenza.
* **Palindromi**: valgono solo per i numeri a 8 cifre. "Beginning 888" e "With 888 On Each Side" = tutto il numero, 888 compreso, è palindromo.
* **Date Club**: il primo o il secondo blocco da 4 cifre è una data valida MMGG. GGMM non conta, e nemmeno una data a cavallo dei blocchi.
* **Years 1900 2015**: le ultime 4 cifre sono un anno tra 1900 e 2015.
* **Numbers Length 7**: numero corto **che non ha nessun'altra categoria**. Per esempio `+888 8 177` sta solo in Repeating 2 Digits.

In [24]:
import re, calendar

def _mmdd(b):
    # True se 4 cifre formano una data valida MMGG (29 febbraio compreso)
    m, d = int(b[:2]), int(b[2:])
    return 1 <= m <= 12 and 1 <= d <= calendar.monthrange(2024, m)[1]

def _pal(x):
    return x == x[::-1]

# s = cifre dopo il prefisso 888, full = tutte le cifre del numero
NUMBER_RULES = {
    'Numbers Repeating 2 Digits': lambda s, full: re.search(r'(\d)\1', s),
    'Numbers Repeating 3 Digits': lambda s, full: re.search(r'(\d)\1{2}', s),
    'Numbers Repeating 4 Digits': lambda s, full: re.search(r'(\d)\1{3}', s),
    'Numbers Repeating 5 Digits': lambda s, full: re.search(r'(\d)\1{4}', s),
    'Numbers Repeating 6 Digits': lambda s, full: re.search(r'(\d)\1{5}', s),
    'Numbers Repeating 7 Digits': lambda s, full: re.search(r'(\d)\1{6}', s),
    'Numbers Repeating 8 Digits': lambda s, full: re.search(r'(\d)\1{7}', s),
    'Numbers Repeating ABAB':     lambda s, full: re.search(r'(\d)(?!\1)(\d)\1\2', s),
    'Numbers Repeating AABB':     lambda s, full: re.search(r'(\d)\1(?!\1)(\d)\2', s),
    'Numbers Repeating ABABAB':   lambda s, full: re.search(r'(\d)(?!\1)(\d)\1\2\1\2', s),
    'Numbers Repeating ABCABC':   lambda s, full: re.search(r'(\d)(?!\1)(\d)(?!\1|\2)(\d)\1\2\3', s),
    'Numbers Repeating AABBCC':   lambda s, full: re.search(r'(\d)\1(?!\1)(\d)\2(?!\1|\2)(\d)\3', s),
    'Numbers Repeating ABCDABCD': lambda s, full: re.search(r'(\d)(?!\1)(\d)(?!\1|\2)(\d)(?!\1|\2|\3)(\d)\1\2\3\4', s),
    'Numbers Repeating AxAxA':    lambda s, full: re.search(r'(\d)\d\1\d\1', s),
    'Numbers Repeating AxAxAxA':  lambda s, full: re.search(r'(\d)\d\1\d\1\d\1', s),
    '69 Club':   lambda s, full: '69' in s,
    '666 Club':  lambda s, full: '666' in s,
    '777 Club':  lambda s, full: '777' in s,
    '888 Club':  lambda s, full: '888' in s,
    '520 Club':  lambda s, full: '520' in s,
    '168 Club':  lambda s, full: '168' in s,
    '1314 Club': lambda s, full: '1314' in s,
    '123 Club':  lambda s, full: '123' in s,
    '007 Club':  lambda s, full: '007' in s,
    'Pi Club':   lambda s, full: '31415' in s,
    # i palindromi valgono solo per i numeri normali (8 cifre dopo 888)
    'Palindromic Numbers':                       lambda s, full: len(s) == 8 and _pal(s),
    'Palindromic Numbers Without First':         lambda s, full: len(s) == 8 and _pal(s[1:]),
    'Palindromic Numbers Without Last':          lambda s, full: len(s) == 8 and _pal(s[:-1]),
    'Palindromic Numbers Beginning 888':         lambda s, full: len(s) == 8 and _pal(full),
    'Palindromic Numbers With 888 On Each Side': lambda s, full: len(s) == 8 and _pal(full),
    'Numbers With 888 On Each Side': lambda s, full: s.endswith('888'),
    'Date Club':        lambda s, full: len(s) == 8 and (_mmdd(s[:4]) or _mmdd(s[4:])),
    'Years 1900 2015':  lambda s, full: len(s) == 8 and 1900 <= int(s[-4:]) <= 2015,
}
NUMBER_CATEGORIES = list(NUMBER_RULES) + ['Numbers Length 7']

def number_categories(name):
    full = re.sub(r'\D', '', name)
    s = full[3:]
    cats = [cat for cat, rule in NUMBER_RULES.items() if rule(s, full)]
    # "Numbers Length 7" solo per i numeri corti che non hanno nessun'altra categoria
    if len(full) == 7 and not cats:
        cats = ['Numbers Length 7']
    return cats

# qualche esempio
for n in ['+888 0777 7777', '+888 0814 3929', '+888 8 177', '+888 8 680', '+888 0768 4929']:
    print(f'{n:16s}', number_categories(n))

+888 0777 7777   ['Numbers Repeating 2 Digits', 'Numbers Repeating 3 Digits', 'Numbers Repeating 4 Digits', 'Numbers Repeating 5 Digits', 'Numbers Repeating 6 Digits', 'Numbers Repeating 7 Digits', 'Numbers Repeating AxAxA', 'Numbers Repeating AxAxAxA', '777 Club', 'Palindromic Numbers Without First']
+888 0814 3929   ['Date Club']
+888 8 177       ['Numbers Repeating 2 Digits']
+888 8 680       ['Numbers Length 7']
+888 0768 4929   []


In [25]:
# ID delle categorie sul sito di Getgems (es. Pi Club -> 160), presi da getgems_categories.csv
cat_ids = pd.read_csv(BASE / 'getgems_categories.csv').query("collezione == 'numbers'")
CATEGORY_ID = dict(zip(cat_ids.categoria, cat_ids.category_id.astype(int).tolist()))
assert set(NUMBER_CATEGORIES) == set(CATEGORY_ID), 'le categorie delle regole e quelle del CSV non coincidono'

numbers_cat = clean['numbers'].copy()
numbers_cat['categorie_getgems'] = numbers_cat['name'].map(number_categories)
numbers_cat['id_categorie_getgems'] = numbers_cat['categorie_getgems'].apply(lambda l: [CATEGORY_ID[c] for c in l])
numbers_cat['n_categorie'] = numbers_cat['categorie_getgems'].str.len()

print('numeri con almeno una categoria:', (numbers_cat.n_categorie > 0).sum(), 'su', len(numbers_cat))
print('quante categorie per numero:', numbers_cat.n_categorie.value_counts().sort_index().to_dict())
conteggi = pd.DataFrame({
    'id_getgems': pd.Series(CATEGORY_ID),
    'numeri': pd.Series([c for l in numbers_cat.categorie_getgems for c in l]).value_counts(),
}).reindex(NUMBER_CATEGORIES).fillna(0).astype(int)
conteggi

numeri con almeno una categoria: 2193 su 5000
quante categorie per numero: {0: 2807, 1: 1830, 2: 284, 3: 31, 4: 20, 5: 12, 6: 6, 7: 8, 9: 1, 10: 1}


,id_getgems,numeri
Numbers Repeating 2 Digits,200,132
Numbers Repeating 3 Digits,201,40
Numbers Repeating 4 Digits,140,24
Numbers Repeating 5 Digits,141,12
Numbers Repeating 6 Digits,142,6
Numbers Repeating 7 Digits,143,1
Numbers Repeating 8 Digits,144,0
Numbers Repeating ABAB,203,117
Numbers Repeating AABB,202,25
Numbers Repeating ABABAB,148,5


### 10b. Verifica sul sito di Getgems

Le verifiche le ho fatte dal browser, guardando le pagine del sito come un normale utente. I risultati sono in due file:
* `verifica_pagine_nft_numeri.csv`: per 67 numeri, l'elenco **completo** delle categorie mostrato sulla pagina dell'NFT
  (50 estratti a caso dal campione, 25 con almeno una categoria e 25 senza, più i 16 numeri corti e `+888 0777 7777`);
* `verifica_categorie_numeri.csv`: una riga per ogni coppia (numero, categoria) controllata. Le fonti sono tre:
  1. **pagina NFT**: le categorie dei 67 numeri qui sopra;
  2. **ricerca nella categoria**: 98 numeri cercati nella pagina di una categoria, compresi alcuni "negativi difficili"
     (es. un numero che finisce con `88` ma non con `888`, l'anno 2041, un `649` per il 69 Club);
  3. **esempi delle pagine delle categorie**: i primi ~15 numeri di ogni categoria (509 in tutto) devono essere riconosciuti dalle regex.

La verifica ha già corretto due regole: i palindromi non valgono per i numeri corti, e "Numbers Length 7" è una categoria di riserva.

In [26]:
# 1) Pagine NFT: l'insieme di categorie previsto deve essere identico a quello mostrato da Getgems
pagine = pd.read_csv(OUT_DIR / 'verifica_pagine_nft_numeri.csv')
pagine['getgems'] = pagine.categorie_getgems.apply(json.loads)
pagine['regex'] = pagine.numero.map(number_categories)
pagine['identico'] = [set(a) == set(b) for a, b in zip(pagine.regex, pagine.getgems)]
print(f"pagine NFT con insieme di categorie identico: {pagine.identico.sum()} / {len(pagine)}",
      f"(di cui {(pagine.getgems.str.len() == 0).sum()} numeri senza nessuna categoria)")

# 2) Tutti i controlli (numero, categoria): pagine NFT + ricerche nelle categorie + esempi delle pagine delle categorie
ver = pd.read_csv(OUT_DIR / 'verifica_categorie_numeri.csv')
# la previsione si ricalcola con le regole attuali: se una regola cambia, la verifica si aggiorna da sola
ver['regex'] = [c in number_categories(n) for n, c in zip(ver.numero, ver.categoria)]
ver['esito'] = [('TP' if g else 'FP') if r else ('FN' if g else 'TN') for r, g in zip(ver.regex, ver.getgems)]
tp, fp, fn = (ver.esito == 'TP').sum(), (ver.esito == 'FP').sum(), (ver.esito == 'FN').sum()
print('controlli (numero, categoria):', len(ver), '|', ver.esito.value_counts().to_dict())
print(f'precision {tp / (tp + fp):.3f} | recall {tp / (tp + fn):.3f}')
print('categorie con almeno un controllo:', ver.categoria.nunique(), 'su', len(NUMBER_CATEGORIES))
pd.crosstab(ver.fonte, ver.esito)

pagine NFT con insieme di categorie identico: 67 / 67 (di cui 25 numeri senza nessuna categoria)
controlli (numero, categoria): 639 | {'TP': 629, 'TN': 10}
precision 1.000 | recall 1.000
categorie con almeno un controllo: 34 su 34


esito,TN,TP
fonte,,
esempio pagina categoria,0,500
pagina NFT,0,57
ricerca nella categoria,10,72


### 10c. Salvataggio del dataset dei numeri con le categorie
`<nome>_clean_categorie.csv` = il dataset pulito dei numeri più tre colonne:
* `categorie_getgems`: elenco JSON di **tutte** le categorie del numero, es. `["69 Club", "Date Club"]`, oppure `[]`;
* `id_categorie_getgems`: gli ID delle stesse categorie sul sito di Getgems, nello stesso ordine, es. `[167, 170]`.
  Il link alla categoria è `https://getgems.io/collection/<collezione>?categoryId=<id>`;
* `n_categorie`: quante sono.

In [27]:
path = OUT_DIR / f"{Path(PATHS['numbers']).stem}_clean_categorie.csv"
(numbers_cat.drop(columns='events')
            .assign(history=numbers_cat.events.apply(json.dumps),
                    categorie_getgems=numbers_cat.categorie_getgems.apply(json.dumps),
                    id_categorie_getgems=numbers_cat.id_categorie_getgems.apply(json.dumps))
            .to_csv(path))
print('salvato', path)

# Rilettura di controllo
chk = pd.read_csv(path, index_col=0)
print(chk.shape, list(chk.columns))
chk[chk.n_categorie > 1][['name', 'categorie_getgems', 'id_categorie_getgems', 'n_categorie']].head()

salvato /Users/cesareanzellotti/Documents/CNA/dataset_clean/fragment_numbers_sample_500_clean_categorie.csv
(5000, 7) ['address', 'name', 'owner_address', 'categorie_getgems', 'id_categorie_getgems', 'n_categorie', 'history']


,name,categorie_getgems,id_categorie_getgems,n_categorie
10,+888 0324 1931,"[""Date Club"", ""Years 1900 2015""]","[170, 169]",2
23,+888 0328 1935,"[""Date Club"", ""Years 1900 2015""]","[170, 169]",2
43,+888 0130 7692,"[""69 Club"", ""Date Club""]","[167, 170]",2
54,+888 0425 1937,"[""Date Club"", ""Years 1900 2015""]","[170, 169]",2
70,+888 0213 7569,"[""69 Club"", ""Date Club""]","[167, 170]",2
